# Descargador de PDFs SECOP-II — Trío piloto

**Documentación metodológica del script `descargar_pdfs_secop_trio.py`**

Equipo de Costeo PDET — ART

---

## Objetivo del documento

Este notebook explica, fase por fase, el funcionamiento del script
**`descargar_pdfs_secop_trio.py`**, que descarga los documentos prioritarios
(Estudios Previos / Anexos Técnicos) de los contratos encontrados en SECOP-II,
organizándolos en `Descargas/{codigo_indicador}/{codigo_contrato}.pdf`. Es un
scraper con **Selenium + Chrome** — no usa ninguna API key, es scraping puro
sobre la página pública del proceso de contratación.

> **⚠ Este script requiere Chrome instalado localmente y confirmaciones
> interactivas (`input()`)**. No es un proceso que se pueda ejecutar de forma
> automática y silenciosa dentro de un notebook batch: la fase de prueba
> (`fase_prueba`) y la confirmación antes de la descarga masiva están
> pensadas para correrse en una sesión interactiva real, viendo el navegador.
> Por eso, en este notebook de documentación **se definen todas las
> funciones pero no se dispara el proceso real** — la celda final queda
> guardada detrás de una bandera (`EJECUTAR = False`) para que el notebook se
> pueda ejecutar de principio a fin sin abrir Chrome, y solo se active cuando
> se vaya a correr de verdad, en Jupyter, de forma interactiva.

## Qué cambia respecto a la versión anterior (v2.5)

La v2.5 descargaba todo suelto a la carpeta de Windows `Downloads`, sin
subcarpetas ni renombrar. Esta versión organiza así:

```
Descargas/
  P5.25/
    CO1.PCCNTR.5771514.pdf
    CO1.PCCNTR.5771514_AT.pdf   <- si hay más de un doc prioritario del mismo contrato
  P5.11/
    CO1.PCCNTR.6569707.pdf
  P3.14/
    CO1.PCCNTR.7701290.pdf
```

Esto se logra **sin mover archivos a mano**: antes de descargar cada
contrato, se le indica a Chrome (vía **Chrome DevTools Protocol — CDP**) que
la carpeta de descarga de ESE contrato es `Descargas/{codigo_indicador}/` —
Chrome descarga directo ahí. Luego solo se renombra el archivo (que ya está
en su carpeta final) de su nombre original de SECOP al código del contrato.

También se agregó un **límite por indicador** (`max_por_indicador`, default
2): por defecto solo se descargan los 2 contratos de mayor valor por
indicador, para no gastar tokens de más en el paso posterior de extracción
de costos con LLM.

> **Generalización de la base de entrada**
>
> Por defecto lee la salida de `buscar_contratos_secop_trio.py`
> (`secop_trio_resultado.xlsx`, hoja `05_Contratos_IDs_URLs`), pero acepta
> **cualquier Excel** que tenga, como mínimo, estas columnas:
>
> - `cod_indicador`, `id_contrato`, `url_proceso` (obligatorias)
> - `referencia`, `nombre_entidad`, `objeto_contrato` (opcionales, solo para
>   el informe final)
> - `validacion_llm` (opcional; solo si se usa `solo_relevantes=True`, para
>   filtrar a los contratos ya marcados como "Relevante" por
>   `validar_contratos_secop_trio.py`)
>
> Es decir: no está atado al trío piloto — cualquier hoja de contratos con
> esas columnas (de cualquier grupo de indicadores) sirve como entrada,
> cambiando `EXCEL_FILE` y `SHEET_NAME` en la configuración.

## Panorama general del pipeline

```
Excel de contratos (cod_indicador, id_contrato, url_proceso, ...)
        │
        ▼
Carga y filtrado (solo_relevantes, max_por_indicador)
        │
        ▼
Fase de prueba interactiva (conectividad, Chrome, detección, descarga de 1 contrato)
        │
        ▼
Confirmación del usuario ──► Descarga masiva (Selenium + CDP por contrato)
        │                           │
        │                           ▼
        │                 Detección y clasificación de documentos (EP / AT / Otro)
        │                           │
        │                           ▼
        │                 Descarga → espera → renombrado a {id_contrato}[.sufijo].pdf
        │
        ▼
Informe Excel (`informe_descargas_PDET.xlsx`, con estado por contrato)
```

## Control global de warnings y errores

Igual que en el notebook del buscador de contratos: el control de warnings y
del comportamiento ante errores se hace en una celda de configuración,
**modificable por celda** (cambiando las variables globales o sobreescribiendo
`warnings.filterwarnings(...)` puntualmente dentro de una celda).

In [ ]:
# ── Control global de warnings y errores (modificable por celda) ────────
import warnings

MOSTRAR_WARNINGS = False   # -> True para ver warnings de pandas/selenium aquí
DETENER_EN_ERROR = False   # -> True para propagar errores inesperados en vez de solo loguearlos

if MOSTRAR_WARNINGS:
    warnings.filterwarnings("default")
else:
    warnings.filterwarnings("ignore")

# Para reactivar warnings SOLO en una celda puntual (sin afectar el resto):
#   with warnings.catch_warnings():
#       warnings.filterwarnings("default")
#       ... código a depurar ...

## Configuración global

Toda la configuración editable vive en las siguientes celdas.

In [ ]:
import os
import re
import sys
import time
import logging
import unicodedata
import subprocess
import platform
from pathlib import Path

import pandas as pd
import requests
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    TimeoutException, NoSuchElementException, SessionNotCreatedException
)

In [ ]:
# ── ENTRADA (GENERAL) ────────────────────────────────────────────────────
# Cualquier Excel con las columnas cod_indicador, id_contrato, url_proceso
# sirve de entrada. Por defecto lee la salida del buscador SECOP:
EXCEL_FILE  = "secop_trio_resultado.xlsx"
SHEET_NAME  = "05_Contratos_IDs_URLs"
REPORT_FILE = "informe_descargas_PDET.xlsx"
LOG_FILE    = "secop_descargador.log"

# Carpeta "Descargas" DENTRO de esta misma carpeta de trabajo — no la carpeta
# Windows Downloads del usuario. Se crea si no existe.
DESCARGAS_DIR = Path("Descargas").resolve()

KILL_CHROME_BEFORE_START = True
HEADLESS_MODE    = False
DOWNLOAD_WAIT    = 30
PAGE_WAIT        = 25
INTERACTIVE_MODE = True   # -> False para saltar todas las confirmaciones input()

SECOP_BASE_URL = "https://community.secop.gov.co" 

In [ ]:
# ── Keywords para clasificar documentos (idéntico al original) ──────────
KEYWORDS_ESTUDIO_PREVIO = [
    "estudio previo", "estudios previos", "estudio de mercado",
    "estudios de mercado", "analisis del sector", "análisis del sector",
]
KEYWORDS_ANEXO_TECNICO = [
    "anexo tecnico", "anexos tecnicos", "anexo técnico", "anexos técnicos",
    "especificacion tecnica", "especificaciones tecnicas",
    "ficha tecnica", "ficha técnica",
    "terminos de referencia", "términos de referencia", "tdr",
]

### Logging

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.FileHandler(LOG_FILE, encoding="utf-8"), logging.StreamHandler()],
)
log = logging.getLogger(__name__)

## Fase 1 — Utilidades

Funciones pequeñas y puras que se reutilizan en el resto del pipeline: normalización de texto (para comparar sin tildes), extracción del `noticeUID` desde la URL de SECOP, nombres de carpeta/archivo seguros para el sistema de archivos, y la clasificación de un documento según su nombre.

In [ ]:
def normalize(text: str) -> str:
    nfkd = unicodedata.normalize("NFKD", str(text))
    return "".join(c for c in nfkd if not unicodedata.combining(c)).lower().strip()


def extract_notice_uid(url: str):
    m = re.search(r'noticeUID=([^&\s]+)', str(url))
    return m.group(1).strip() if m else None


def carpeta_indicador(cod_indicador: str) -> str:
    """Nombre de subcarpeta seguro para el sistema de archivos: 'P5.25.' -> 'P5.25'."""
    limpio = str(cod_indicador).strip().rstrip(".")
    return re.sub(r'[\\/*?:"<>|]', "_", limpio)


def nombre_archivo_seguro(id_contrato: str) -> str:
    """Código de contrato como nombre de archivo, sin caracteres inválidos en Windows."""
    return re.sub(r'[\\/*?:"<>|]', "_", str(id_contrato).strip())


def classify_document(filename: str) -> tuple:
    """Retorna (prioridad, etiqueta). 1=EP, 2=AT, 9=Otro."""
    n = normalize(filename)
    for kw in KEYWORDS_ESTUDIO_PREVIO:
        if kw in n:
            return 1, "EP"
    for kw in KEYWORDS_ANEXO_TECNICO:
        if kw in n:
            return 2, "AT"
    return 9, "OT"


def confirm(prompt: str) -> bool:
    if not INTERACTIVE_MODE:
        return True
    resp = input(f"\n{prompt} [s/N]: ").strip().lower()
    return resp in ("s", "si", "sí", "y", "yes")


def sep(char="═", title=""):
    w = 70
    if title:
        t = f" {title} "
        p = (w - len(t)) // 2
        print(char * p + t + char * (w - p - len(t)))
    else:
        print(char * w)

`classify_document` clasifica por **prioridad**: 1 = Estudio Previo (el más buscado), 2 = Anexo Técnico, 9 = cualquier otro documento. Esta prioridad es la que después decide qué se descarga primero y cuáles documentos son "prioritarios" en `process_contract`.

## Fase 2 — Carga de contratos

In [ ]:
def load_contracts(excel_path: str, hoja: str, solo_relevantes: bool = False,
                    max_por_indicador: int = None) -> pd.DataFrame:
    if not os.path.exists(excel_path):
        log.error(f"❌ Archivo no encontrado: {os.path.abspath(excel_path)}")
        return pd.DataFrame()

    log.info(f"📂 Leyendo: {excel_path} | hoja: {hoja}")
    df = pd.read_excel(excel_path, sheet_name=hoja)
    log.info(f"   {len(df)} filas en el Excel")

    # Acepta tanto la salida de buscar_contratos_secop_trio.py (05_Contratos_IDs_URLs)
    # como la de validar_contratos_secop_trio.py (Contratos_Validados_LLM) —
    # los nombres de columna son los mismos en ambas.
    required = ["cod_indicador", "id_contrato", "url_proceso"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        log.error(f"❌ Columnas faltantes: {missing}")
        return pd.DataFrame()

    if solo_relevantes:
        if "validacion_llm" not in df.columns:
            log.warning("⚠️  --solo-relevantes pedido pero la hoja no trae 'validacion_llm' — se ignora el filtro")
        else:
            antes = len(df)
            df = df[df["validacion_llm"] == "Relevante"].copy()
            log.info(f"   Filtrado a solo 'Relevante': {antes} → {len(df)} filas")

    if max_por_indicador:
        # Preserva el orden que ya trae la hoja de origen (05_Contratos_IDs_URLs
        # viene ordenada por valor_contrato_cop DESC; Contratos_Validados_LLM
        # viene ordenada por validacion_llm DESC, valor DESC) — al tomar solo
        # las primeras N filas de cada grupo, nos quedamos con los contratos
        # de mayor valor/relevancia, no con los primeros que aparezcan al azar.
        antes = len(df)
        df = df.groupby("cod_indicador", group_keys=False).head(max_por_indicador)
        log.info(f"   Limitado a {max_por_indicador} por indicador: {antes} → {len(df)} filas")

    records = []
    for _, row in df.iterrows():
        url = str(row.get("url_proceso", "")).strip()
        if not url.startswith("http"):
            continue
        uid = extract_notice_uid(url)
        if not uid:
            continue
        id_contrato = str(row.get("id_contrato", "")).strip()
        if not id_contrato:
            continue
        records.append({
            "cod_indicador": str(row["cod_indicador"]).strip(),
            "id_contrato":   id_contrato,
            "referencia":    str(row.get("referencia", ""))[:60],
            "entidad":       str(row.get("nombre_entidad", ""))[:80],
            "objeto":        str(row.get("objeto_contrato", ""))[:200],
            "url":           url,
            "notice_uid":    uid,
        })

    df_out = pd.DataFrame(records).drop_duplicates(subset=["id_contrato"]).reset_index(drop=True)
    log.info(f"   {len(df_out)} contratos únicos con URL válida")
    return df_out

Puntos clave de esta función:

- **Valida las columnas mínimas** (`cod_indicador`, `id_contrato`, `url_proceso`) y aborta con un error claro si faltan, en vez de fallar más adelante con un traceback confuso.
- El filtro `solo_relevantes` es **opcional y tolerante**: si la hoja no trae `validacion_llm`, se avisa y se ignora el filtro en vez de romper la carga.
- El límite `max_por_indicador` se aplica **después** de cualquier filtro, y confía en que la hoja de origen ya viene ordenada por valor o relevancia — así los primeros N de cada grupo son los contratos más importantes, no una muestra aleatoria.
- Se descartan filas sin URL válida o sin `noticeUID` extraíble, y se eliminan duplicados por `id_contrato`.

## Fase 3 — Control de Chrome (Selenium + CDP)

In [ ]:
def kill_chrome():
    if platform.system() == "Windows":
        for proc in ["chrome.exe", "chromedriver.exe"]:
            subprocess.run(["taskkill", "/F", "/IM", proc, "/T"], capture_output=True, timeout=10)
        time.sleep(2)


def build_driver() -> "webdriver.Chrome":
    """
    Abre Chrome. La carpeta de descarga inicial es Descargas/ (la base);
    cada contrato la redirige a su propia subcarpeta vía CDP antes de bajar.
    """
    DESCARGAS_DIR.mkdir(parents=True, exist_ok=True)
    opt = Options()
    opt.add_argument("--no-sandbox")
    opt.add_argument("--disable-dev-shm-usage")

    if HEADLESS_MODE:
        opt.add_argument("--headless=new")
        opt.add_argument("--window-size=1920,1080")

    opt.add_experimental_option("prefs", {
        "download.default_directory":                              str(DESCARGAS_DIR),
        "download.prompt_for_download":                            False,
        "download.directory_upgrade":                              True,
        "download.open_pdf_in_system_reader":                      False,
        "plugins.always_open_pdf_externally":                      True,
        "profile.default_content_setting_values.automatic_downloads": 1,
        "safebrowsing.enabled":                                    True,
    })

    for intento in range(3):
        try:
            drv = webdriver.Chrome(options=opt)
            try:
                drv.execute_cdp_cmd("Page.setDownloadBehavior", {
                    "behavior": "allow", "downloadPath": str(DESCARGAS_DIR),
                })
            except Exception:
                pass
            log.info("   ✅ Chrome abierto correctamente")
            return drv
        except SessionNotCreatedException as e:
            log.warning(f"   Intento {intento+1}/3 falló: {str(e)[:80]}")
            kill_chrome()
            time.sleep(3)

    raise RuntimeError("No se pudo abrir Chrome tras 3 intentos")


def set_download_dir(driver, target_dir: Path):
    """Redirige la carpeta de descarga de Chrome a target_dir vía CDP (sin reabrir el navegador)."""
    target_dir.mkdir(parents=True, exist_ok=True)
    driver.execute_cdp_cmd("Page.setDownloadBehavior", {
        "behavior": "allow", "downloadPath": str(target_dir.resolve()),
    })

`set_download_dir` es la pieza clave de la organización en subcarpetas:
se llama **una vez por contrato**, justo antes de navegar a su página, y le
dice a Chrome (vía CDP, sin reabrir el navegador ni perder la sesión) que
descargue ahí. Así se evita mover archivos manualmente después — Chrome
descarga directo en la carpeta final del indicador.

## Fase 4 — Espera de descarga y renombrado

In [ ]:
def wait_for_download(watch_dir: Path, before_set: set, timeout=DOWNLOAD_WAIT):
    deadline = time.time() + timeout
    while time.time() < deadline:
        time.sleep(1)
        try:
            current = set(watch_dir.glob("*"))
        except Exception:
            continue
        nuevos = current - before_set
        en_progreso = [f for f in nuevos if f.suffix == ".crdownload"]
        completados = [f for f in nuevos if f.suffix not in (".crdownload", ".tmp")]
        if completados:
            f = max(completados, key=lambda x: x.stat().st_mtime)
            try:
                s1 = f.stat().st_size
                time.sleep(1.5)
                s2 = f.stat().st_size
                if s2 >= s1 and s2 > 0:
                    return f
            except Exception:
                continue
        elif en_progreso:
            deadline = max(deadline, time.time() + 5)
    return None


def renombrar_a_codigo_contrato(archivo: Path, id_contrato: str, sufijo: str = "") -> Path:
    """Renombra el PDF recién descargado a {id_contrato}[_sufijo].ext, evitando colisiones."""
    base = nombre_archivo_seguro(id_contrato) + (f"_{sufijo}" if sufijo else "")
    destino = archivo.with_name(base + archivo.suffix)
    contador = 2
    while destino.exists() and destino != archivo:
        destino = archivo.with_name(f"{base}_{contador}{archivo.suffix}")
        contador += 1
    archivo.rename(destino)
    return destino

`wait_for_download` no solo espera a que aparezca un archivo nuevo:
verifica que **no sea un `.crdownload`** (descarga en progreso) y que su
tamaño se **estabilice** entre dos lecturas (1.5s de diferencia) antes de
darlo por completo — esto evita renombrar un archivo que todavía se está
escribiendo. Si hay una descarga en progreso, el plazo de espera se extiende
dinámicamente (`deadline = max(deadline, time.time() + 5)`) en vez de cortar
a mitad de una descarga real.

## Fase 5 — Detección de documentos en la página

In [ ]:
def get_document_list(driver) -> list:
    docs = []
    idx = 0
    while True:
        try:
            name_el = driver.find_element(By.ID, f"tdColumnDocumentNameP2Gen_spnDocumentName_{idx}")
            btn_el = driver.find_element(By.ID, f"lnkDownloadLinkP3Gen_{idx}")
            filename = name_el.text.strip()
            if filename:
                pri, label = classify_document(filename)
                docs.append({"filename": filename, "priority": pri, "label": label, "element": btn_el})
        except NoSuchElementException:
            break
        idx += 1
    return sorted(docs, key=lambda d: d["priority"])

Recorre los elementos de la tabla de documentos de SECOP por índice incremental (`_0`, `_1`, `_2`, ...) hasta que uno no exista — no depende de saber de antemano cuántos documentos tiene el proceso. El resultado queda **ordenado por prioridad** (Estudio Previo primero, luego Anexo Técnico, luego el resto), listo para que `process_contract` decida qué descargar.

## Fase 6 — Fase de prueba interactiva

In [ ]:
def fase_prueba(test_url: str, test_cod_indicador: str, test_id_contrato: str) -> bool:
    sep("═", "🧪 FASE DE PRUEBA")
    test_dir = DESCARGAS_DIR / carpeta_indicador(test_cod_indicador)

    sep("─", "TEST 1: Conectividad HTTP")
    try:
        r = requests.get(SECOP_BASE_URL, headers={"User-Agent": "Mozilla/5.0"}, timeout=15)
        if r.status_code == 200 and "secop" in r.text.lower():
            print(f"\n   ✅ SECOP-II responde correctamente ({r.elapsed.total_seconds():.2f}s)")
        else:
            print(f"\n   ❌ HTTP {r.status_code}")
            return False
    except Exception as e:
        print(f"\n   ❌ Sin conexión: {e}")
        return False

    if not confirm("Test 1 OK. ¿Abrir Chrome?"):
        return False

    driver = None
    try:
        sep("─", "TEST 2: Abrir Chrome y cargar SECOP")
        print("\n🚀 Abriendo Chrome...")
        driver = build_driver()
        set_download_dir(driver, test_dir)

        driver.get(test_url)
        try:
            WebDriverWait(driver, PAGE_WAIT).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
        except TimeoutException:
            pass
        time.sleep(5)

        body_len = len(driver.find_element(By.TAG_NAME, "body").text)
        print(f"   {'✅' if body_len>500 else '⚠️ '} Página cargada ({body_len:,} caracteres)"
              + ("" if body_len > 500 else " — puede haber CAPTCHA"))

        if not confirm("¿Continuar con Test 3 (detectar documentos)?"):
            driver.quit(); return False

        sep("─", "TEST 3: Detección de documentos")
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        driver.execute_script("window.scrollTo(0, 0);")
        time.sleep(1)

        docs = get_document_list(driver)
        print(f"\n   Documentos detectados: {len(docs)}")
        for i, d in enumerate(docs[:8], 1):
            print(f"   {i:2d}. [{d['label']:3s}] {d['filename'][:55]}")
        if not docs:
            print("   ⚠️  Sin documentos (posible CAPTCHA — resuélvelo en Chrome)")

        if not confirm("¿Continuar con Test 4 (descarga real + renombrado)?"):
            driver.quit(); return False

        sep("─", "TEST 4: Descarga de prueba con carpeta + nombre final")
        if not docs:
            print("   ⏭️  Sin documentos para probar — saltando")
            driver.quit()
            return confirm("¿Continuar igualmente con el proceso masivo?")

        doc_test = docs[0]
        print(f"\n   Descargando: {doc_test['filename'][:60]}")
        print(f"   Carpeta destino: {test_dir}")

        before = set(test_dir.glob("*")) if test_dir.exists() else set()
        driver.execute_script("arguments[0].click();", doc_test["element"])
        nuevo = wait_for_download(test_dir, before)

        ok = False
        if nuevo:
            final = renombrar_a_codigo_contrato(nuevo, test_id_contrato)
            kb = final.stat().st_size / 1024
            print(f"   ✅ Descargado y renombrado: {final.relative_to(DESCARGAS_DIR)} ({kb:.0f} KB)")
            ok = True
        else:
            print(f"   ❌ Timeout — el archivo no apareció en {DOWNLOAD_WAIT}s")

        driver.quit()
        driver = None

        sep("─")
        print(f"  ✅  Test 1 HTTP       — OK")
        print(f"  {'✅' if body_len>500 else '⚠️ '}  Test 2 Chrome     — {'OK' if body_len>500 else 'Página vacía'}")
        print(f"  {'✅' if docs else '⚠️ '}  Test 3 Documentos — {len(docs)} detectados")
        print(f"  {'✅' if ok else '❌'}  Test 4 Descarga   — {'OK' if ok else 'FALLÓ'}")

        if ok:
            print("\n🎉 Prueba exitosa — listo para descarga masiva")
            return True
        return confirm("⚠️  La descarga de prueba falló. ¿Continuar igual?")

    except Exception as e:
        log.error(f"Error en fase de prueba: {e}")
        if driver:
            try: driver.quit()
            except Exception: pass
        return False

Antes de lanzar la descarga masiva (que puede tomar horas), se corre
esta prueba de 4 pasos sobre **un solo contrato**, con confirmación manual
entre cada paso:

1. **Conectividad HTTP** — ¿responde SECOP-II?
2. **Chrome** — ¿se abre el navegador y carga la página?
3. **Detección de documentos** — ¿se encuentran documentos en la tabla?
4. **Descarga real** — ¿se descarga y renombra correctamente un archivo?

Si algo falla a mitad de camino, se puede decidir seguir igual (`confirm`)
en vez de que el script aborte de forma rígida — útil, por ejemplo, si el
proceso de prueba no tiene documentos pero se sabe que otros sí.

## Fase 7 — Procesar un contrato

In [ ]:
def process_contract(driver, cod_indicador: str, id_contrato: str, notice_uid: str,
                      url: str, entidad: str, objeto: str) -> dict:
    """
    Navega al contrato, redirige la descarga a Descargas/{cod_indicador}/,
    descarga los documentos prioritarios y los renombra a {id_contrato}[.pdf|_AT.pdf|...].
    """
    target_dir = DESCARGAS_DIR / carpeta_indicador(cod_indicador)
    target_dir.mkdir(parents=True, exist_ok=True)

    result = {
        "cod_indicador": cod_indicador, "id_contrato": id_contrato, "notice_uid": notice_uid,
        "url": url, "entidad": entidad, "objeto": objeto[:150],
        "estado": "pendiente", "total_docs_pagina": 0, "docs_descargados": 0,
        "tiene_estudio_previo": "No", "tiene_anexo_tecnico": "No",
        "archivos_descargados": "", "carpeta": str(target_dir.relative_to(DESCARGAS_DIR)), "error": "",
    }

    try:
        set_download_dir(driver, target_dir)

        driver.get(url)
        try:
            WebDriverWait(driver, PAGE_WAIT).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
        except TimeoutException:
            pass
        time.sleep(4)
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        driver.execute_script("window.scrollTo(0, 0);")
        time.sleep(1)

        docs = get_document_list(driver)
        result["total_docs_pagina"] = len(docs)
        if not docs:
            result["estado"] = "sin_documentos"
            return result

        prioritarios = [d for d in docs if d["priority"] in (1, 2)]
        a_descargar = prioritarios if prioritarios else docs[:3]
        multiples = len(a_descargar) > 1

        descargados = []
        for doc in a_descargar:
            before = set(target_dir.glob("*"))
            try:
                driver.execute_script("arguments[0].click();", doc["element"])
            except Exception:
                continue

            nuevo = wait_for_download(target_dir, before)
            if nuevo:
                sufijo = doc["label"] if multiples else ""
                final = renombrar_a_codigo_contrato(nuevo, id_contrato, sufijo)
                descargados.append(final.name)
                log.info(f"  ✓ [{doc['label']}] {final.name}")
                if doc["priority"] == 1:
                    result["tiene_estudio_previo"] = "Sí"
                elif doc["priority"] == 2:
                    result["tiene_anexo_tecnico"] = "Sí"
            else:
                log.warning(f"  ✗ Timeout: {doc['filename'][:50]}")
            time.sleep(2)

        result["docs_descargados"] = len(descargados)
        result["archivos_descargados"] = " | ".join(descargados)
        result["estado"] = "exitoso" if descargados else "descarga_fallida"

    except Exception as e:
        result["estado"] = "error"
        result["error"] = str(e)[:200]
        log.error(f"  Error en {id_contrato}: {e}")

    return result

Lógica de selección de documentos a descargar:

- Si hay documentos **prioritarios** (Estudio Previo o Anexo Técnico), se
  descargan **todos los prioritarios** (no solo el primero).
- Si **no** hay ninguno prioritario, se descargan los **primeros 3** de la
  lista como respaldo, para no quedarse sin ningún insumo del contrato.
- Si se descarga más de un documento del mismo contrato, cada archivo se
  renombra con un **sufijo** (`_EP`, `_AT`, ...) para no pisarse entre sí.

Cualquier error durante el proceso (timeout, elemento no encontrado, etc.)
queda capturado y registrado en el resultado (`estado="error"`) — nunca
interrumpe el procesamiento de los demás contratos del lote.

## Fase 8 — Informe Excel

In [ ]:
def generate_report(results: list, path: str):
    wb = Workbook()
    ws = wb.active
    ws.title = "Descargas"

    headers = [
        "Cod. Indicador", "Id Contrato", "Notice UID", "Estado", "Carpeta",
        "Docs en Página", "Docs Descargados", "Tiene EP", "Tiene AT",
        "Archivos Descargados", "Entidad", "Objeto", "URL", "Error"
    ]
    for ci, h in enumerate(headers, 1):
        ws.cell(1, ci, h).font = Font(bold=True)

    colores = {"exitoso": "C6EFCE", "sin_documentos": "FFEB9C",
               "descarga_fallida": "FFC7CE", "error": "FFC7CE"}
    for ri, r in enumerate(results, 2):
        vals = [
            r["cod_indicador"], r["id_contrato"], r["notice_uid"], r["estado"], r["carpeta"],
            r["total_docs_pagina"], r["docs_descargados"], r["tiene_estudio_previo"], r["tiene_anexo_tecnico"],
            r["archivos_descargados"], r.get("entidad", ""), r.get("objeto", ""), r["url"], r["error"],
        ]
        fill = PatternFill("solid", start_color=colores.get(r["estado"], "F2F2F2"))
        for ci, v in enumerate(vals, 1):
            cell = ws.cell(ri, ci, v)
            cell.font = Font(size=9)
            if ci == 4:
                cell.fill = fill

    ws.freeze_panes = "A2"
    wb.save(path)
    log.info(f"📊 Informe guardado: {path}")

El informe usa **color por estado** en la columna "Estado" (verde = exitoso, amarillo = sin documentos, rojo = descarga fallida o error) para que se pueda escanear visualmente qué contratos necesitan revisión manual, sin tener que leer fila por fila.

## Fase 9 — Proceso principal (`main`)

En el script original esto se maneja por línea de comandos (`argparse`), con
las opciones:

- `--excel` (default `secop_trio_resultado.xlsx`)
- `--hoja` (default `05_Contratos_IDs_URLs`)
- `--solo-relevantes` (filtra a `validacion_llm == "Relevante"`, si esa
  columna existe)
- `--max-por-indicador` (default 2; usar 0 para no limitar)

En el notebook, `main()` recibe los mismos parámetros como argumentos de
función en vez de flags de CLI.

In [ ]:
def main(excel=EXCEL_FILE, hoja=SHEET_NAME, solo_relevantes=False, max_por_indicador=2):
    max_por_indicador = max_por_indicador if max_por_indicador and max_por_indicador > 0 else None

    sep("═", "🚀 SECOP-II DOWNLOADER — Descargas/{indicador}/{contrato}.pdf")
    print()
    log.info("Iniciando proceso")

    try:
        import selenium
        print(f"📊 Selenium {selenium.__version__}  |  Python {sys.version.split()[0]}")
        major, minor = map(int, selenium.__version__.split(".")[:2])
        if major < 4 or (major == 4 and minor < 11):
            print("⚠️  Selenium desactualizado. Ejecuta: py -m pip install --upgrade selenium")
    except Exception:
        pass

    chrome_paths = [
        r"C:\Program Files\Google\Chrome\Application\chrome.exe",
        r"C:\Program Files (x86)\Google\Chrome\Application\chrome.exe",
        os.path.expanduser(r"~\AppData\Local\Google\Chrome\Application\chrome.exe"),
    ]
    chrome_ok = any(os.path.exists(p) for p in chrome_paths)
    print(f"🌐 Chrome: {'✅ encontrado' if chrome_ok else '❌ no encontrado'}")
    print(f"📥 Carpeta base de descarga: {DESCARGAS_DIR}")

    if not chrome_ok:
        print("   Instala Chrome desde https://www.google.com/chrome/")
        return

    if KILL_CHROME_BEFORE_START:
        kill_chrome()

    df = load_contracts(excel, hoja, solo_relevantes=solo_relevantes,
                         max_por_indicador=max_por_indicador)
    if df.empty:
        return

    print(f"\n📋 {len(df)} contratos cargados"
          + (f" (máx. {max_por_indicador} por indicador)" if max_por_indicador else " (sin límite por indicador)"))
    print(f"\n{'Código':12s} {'Contratos':>10}")
    print("-" * 24)
    for ind, grupo in df.groupby("cod_indicador"):
        print(f"  {ind:12s} {len(grupo):>6}")

    primero = df.iloc[0]
    if not fase_prueba(primero["url"], primero["cod_indicador"], primero["id_contrato"]):
        print("\n⏹️  Proceso detenido")
        return

    sep()
    print(f"\n📌 Los archivos quedarán organizados así:")
    print(f"   {DESCARGAS_DIR}\\{{codigo_indicador}}\\{{codigo_contrato}}.pdf")
    print(f"\n   Tiempo estimado: ~{len(df) * 20 // 60} minutos")

    if not confirm(f"\n¿Iniciar descarga masiva de {len(df)} contratos?"):
        return

    print("\n🌐 Abriendo Chrome para proceso masivo...")
    driver = build_driver()

    results = []
    total = len(df)
    t0 = time.time()

    for idx, row in df.iterrows():
        num = idx + 1
        elapsed = time.time() - t0
        eta_min = ((elapsed / num) * (total - num)) / 60 if num > 0 else 0

        log.info(f"\n[{num}/{total}] {row['cod_indicador']} | {row['id_contrato']} | ETA: {eta_min:.1f} min")
        print(f"\n  [{num:02d}/{total}] {row['cod_indicador']} | {row['id_contrato']} — {row['entidad'][:40]}")

        res = process_contract(
            driver,
            cod_indicador=row["cod_indicador"], id_contrato=row["id_contrato"],
            notice_uid=row["notice_uid"], url=row["url"],
            entidad=row["entidad"], objeto=row["objeto"],
        )
        results.append(res)

        if num % 10 == 0:
            generate_report(results, REPORT_FILE.replace(".xlsx", f"_parcial_{num}.xlsx"))
            print(f"  💾 Informe parcial guardado ({num}/{total})")

        time.sleep(2)

    driver.quit()
    generate_report(results, REPORT_FILE)

    df_res = pd.DataFrame(results)
    exitosos = (df_res["estado"] == "exitoso").sum()
    sin_docs = (df_res["estado"] == "sin_documentos").sum()
    fallidos = (df_res["estado"] == "descarga_fallida").sum()
    errores = (df_res["estado"] == "error").sum()
    total_docs = df_res["docs_descargados"].sum()

    sep("═", "✅ PROCESO COMPLETADO")
    print(f"\n  Total contratos procesados:  {total}")
    print(f"  Exitosos:                    {exitosos}")
    print(f"  Sin documentos en SECOP:     {sin_docs}")
    print(f"  Descarga fallida:            {fallidos}")
    print(f"  Error de navegación:         {errores}")
    print(f"  Total archivos descargados:  {total_docs}")
    print(f"\n  📥 Archivos en: {DESCARGAS_DIR}\\{{codigo_indicador}}\\{{codigo_contrato}}.pdf")
    print(f"  📊 Informe en:  {REPORT_FILE}")

Nótese el **informe parcial cada 10 contratos**
(`REPORT_FILE.replace(".xlsx", f"_parcial_{num}.xlsx")`) — en un lote de
cientos de contratos, si el proceso se interrumpe a la mitad (Chrome se
cierra, se pierde la conexión, etc.) no se pierde el trabajo ya hecho: queda
un Excel parcial guardado con lo procesado hasta ese punto.

## Fase 10 — Ejecución

Esta celda queda **desactivada por defecto** (`EJECUTAR = False`) para que el
notebook se pueda correr de principio a fin sin abrir Chrome ni bloquearse
en un `input()`. Para una corrida real:

1. Cambiar `EJECUTAR = True`.
2. Correr esta celda **en una sesión interactiva de Jupyter** (no vía
   ejecución automática por lotes) — el proceso abre Chrome de verdad y pide
   confirmaciones en pantalla durante la fase de prueba.
3. Ajustar `EXCEL`, `HOJA`, `SOLO_RELEVANTES` y `MAX_POR_INDICADOR` según el
   grupo de contratos que se quiera descargar en esa corrida.

In [ ]:
EJECUTAR          = False   # -> True para correr el proceso real (requiere Chrome + sesión interactiva)
EXCEL             = EXCEL_FILE
HOJA              = SHEET_NAME
SOLO_RELEVANTES   = False
MAX_POR_INDICADOR = 2       # -> 0 para no limitar contratos por indicador

if EJECUTAR:
    try:
        main(excel=EXCEL, hoja=HOJA, solo_relevantes=SOLO_RELEVANTES,
             max_por_indicador=MAX_POR_INDICADOR)
    except KeyboardInterrupt:
        print("\n\n⏹️  Interrumpido por el usuario (Ctrl+C)")
    except Exception as e:
        log.exception(f"Error fatal: {e}")
        print(f"\n❌ ERROR FATAL: {e}")
        if DETENER_EN_ERROR:
            raise
else:
    print("EJECUTAR=False — el proceso real no se disparó. "
          "Cambiar a True y correr esta celda en Jupyter interactivo para descargar de verdad.")